# Stage 1: CyTOF preprocessing

This notebook runs the source notebook's loading, CD45/GFP filtering, raw histone gating, and arcsinh transformation functions from the intended checkout. It reads only the authoritative annotated raw CSV directory and writes to the explicitly selected run directory.

In [1]:
from pathlib import Path
import sys

# Support launching Jupyter from either the checkout root or its notebooks folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
# Prefer this checkout over another installed copy or worktree of app.
sys.path.insert(0, str(PROJECT_ROOT))

import app  # noqa: E402

print(f"Kernel executable: {sys.executable}")
print(f"Working directory: {Path.cwd().resolve()}")
print(f"Imported app package: {Path(app.__file__).resolve()}")

# Fail early if Python resolved app from outside this checkout.
if Path(app.__file__).resolve().parent != PROJECT_ROOT / "app":
    raise RuntimeError("app was imported from outside the intended checkout")

Kernel executable: /Users/yishai/Desktop/epigenetics_field_theory/.venv/bin/python
Working directory: /Users/yishai/Desktop/epigenetics_field_theory/notebooks
Imported app package: /Users/yishai/Desktop/epigenetics_field_theory/app/__init__.py


In [2]:
from app.dotenv import OUTPUT_ROOT, data_dir
from app.preprocessing import (
    arcsinh_transform,
    load_raw_data,
    plot_cd45_histograms,
    plot_gfp_histograms,
    process_arcsinh_files,
    process_gated_files,
    save_gfp_split,
    save_loading_summary,
    split_cells_by_gfp,
)
import pandas as pd
from IPython.display import display

In [3]:
INPUT_DIR = data_dir
EXPERIMENTS = ["batch2", "batch3", "batch4_extra_treatment"]
CD45_CUTOFFS_ASINH = {experiment: 2.0 for experiment in EXPERIMENTS}
GFP_CUTOFFS_ASINH = {"batch2": 2.0, "batch3": 4.0, "batch4_extra_treatment": 3.0}

In [4]:
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Raw data folder not found: {INPUT_DIR}")
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_ROOT}")

Input directory: /Users/yishai/Desktop/epigenetics_field_theory/raw_dats
Output directory: /Users/yishai/Desktop/epigenetics_field_theory/results/experiment_141026


## 1. Raw data loading

In [5]:
DF_loaded = load_raw_data(INPUT_DIR, EXPERIMENTS)
loading_summary = save_loading_summary(DF_loaded, OUTPUT_ROOT / "1. raw data")
print(f"Loaded cells: {len(DF_loaded)}")
display(loading_summary)

Loaded cells: 990583


,experiment,condition,sample_name,replicate,source_file,source_path,n_cells,saved_file,first_cell_id,last_cell_id
0,batch2,treated,batch2_1473_gem,batch2_1473_gem,batch2_1473_gem.csv,C:\Users\micha\OneDrive\Desktop\allCyTOFsDATA\...,13455,,batch2_cell_00000001,batch2_cell_00013455
1,batch2,treated,batch2_1484_gem,batch2_1484_gem,batch2_1484_gem.csv,C:\Users\micha\OneDrive\Desktop\allCyTOFsDATA\...,14359,,batch2_cell_00032938,batch2_cell_00047296
2,batch2,treated,batch2_1494_gem,batch2_1494_gem,batch2_1494_gem.csv,C:\Users\micha\OneDrive\Desktop\allCyTOFsDATA\...,12884,,batch2_cell_00069639,batch2_cell_00082522
3,batch2,untreated,batch2_1474_unt,batch2_1474_unt,batch2_1474_unt.csv,C:\Users\micha\OneDrive\Desktop\allCyTOFsDATA\...,19482,,batch2_cell_00013456,batch2_cell_00032937
4,batch2,untreated,batch2_1485_unt,batch2_1485_unt,batch2_1485_unt.csv,C:\Users\micha\OneDrive\Desktop\allCyTOFsDATA\...,22342,,batch2_cell_00047297,batch2_cell_00069638
5,batch2,untreated,batch2_1610_unt,batch2_1610_unt,batch2_1610_unt.csv,C:\Users\micha\OneDrive\Desktop\allCyTOFsDATA\...,17378,,batch2_cell_00082523,batch2_cell_00099900
6,batch3,treated,batch3_1493_gem,batch3_1493_gem,batch3_1493_gem.csv,C:\Users\micha\OneDrive\Desktop\allCyTOFsDATA\...,47276,,batch3_cell_00287323,batch3_cell_00334598
7,batch3,treated,batch3_1496_gem,batch3_1496_gem,batch3_1496_gem.csv,C:\Users\micha\OneDrive\Desktop\allCyTOFsDATA\...,103425,,batch3_cell_00334599,batch3_cell_00438023
8,batch3,treated,batch3_1497_gem,batch3_1497_gem,batch3_1497_gem.csv,C:\Users\micha\OneDrive\Desktop\allCyTOFsDATA\...,96098,,batch3_cell_00438024,batch3_cell_00534121
9,batch3,treated,batch3_1498_gem,batch3_1498_gem,batch3_1498_gem.csv,C:\Users\micha\OneDrive\Desktop\allCyTOFsDATA\...,70128,,batch3_cell_00534122,batch3_cell_00604249


## 2. CD45 and GFP filtering

In [6]:
PLOT_ROOT = OUTPUT_ROOT / "2. CD45 + GFP gating plots"
plot_cd45_histograms(INPUT_DIR, EXPERIMENTS, PLOT_ROOT / "CD45 histograms", CD45_CUTOFFS_ASINH)
gfp_histogram_summary = plot_gfp_histograms(
    DF_loaded, EXPERIMENTS, GFP_CUTOFFS_ASINH, PLOT_ROOT / "GFP histograms"
)
gfp_histogram_summary.to_csv(PLOT_ROOT / "GFP_histogram_summary.csv", index=False)
display(gfp_histogram_summary)
gfp_splits, gfp_split_summary = split_cells_by_gfp(
    DF_loaded, EXPERIMENTS, GFP_CUTOFFS_ASINH, cd45_cutoff=2.0
)
split_files = save_gfp_split(gfp_splits, OUTPUT_ROOT / "3. GFP split after CD45 exclusion")
gfp_split_summary.to_csv(
    OUTPUT_ROOT / "3. GFP split after CD45 exclusion" / "GFP_split_summary.csv", index=False
)
print(f"Refreshed split files: {len(split_files)}")
display(gfp_split_summary)

,experiment,cd45_cutoff_asinh,gfp_cutoff_asinh_shown,n_total_valid,n_cd45neg_kept,n_cd45pos_excluded,plot_file
0,batch2,2.0,2.0,99900,99613,287,/Users/yishai/Desktop/epigenetics_field_theory...
1,batch3,2.0,4.0,604249,561469,42780,/Users/yishai/Desktop/epigenetics_field_theory...
2,batch4_extra_treatment,2.0,3.0,286434,284725,1709,/Users/yishai/Desktop/epigenetics_field_theory...


Refreshed split files: 34


,experiment,sample_name,n_GFP_high,n_GFP_low
0,batch2,batch2_1473_gem,2656,10768
1,batch2,batch2_1484_gem,960,13361
2,batch2,batch2_1494_gem,2705,10150
3,batch2,batch2_1474_unt,4387,15060
4,batch2,batch2_1485_unt,5533,16709
5,batch2,batch2_1610_unt,3987,13337
6,batch3,batch3_1493_gem,4269,40328
7,batch3,batch3_1496_gem,13944,82197
8,batch3,batch3_1497_gem,13762,75265
9,batch3,batch3_1498_gem,5144,62034


## 3. Raw histone gating

In [7]:
SPLIT_ROOT = OUTPUT_ROOT / "3. GFP split after CD45 exclusion"
GATED_ROOT = OUTPUT_ROOT / "4. data processing after GFP split" / "GATED_a"
gating_summary = process_gated_files(SPLIT_ROOT, GATED_ROOT, EXPERIMENTS)
display(gating_summary)

,experiment,condition,sample_name,gfp_group,input_file,gated_file,gate_columns,gate_rule,n_input_cells,n_gated_cells,n_removed_by_gate,pct_kept
0,batch2,treated,batch2_1473_gem,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,H3.3 > 5 and H4 > 5 on raw values,2656,1638,1018,61.671687
1,batch2,untreated,batch2_1474_unt,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,H3.3 > 5 and H4 > 5 on raw values,4387,3399,988,77.478915
2,batch2,treated,batch2_1484_gem,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,H3.3 > 5 and H4 > 5 on raw values,960,681,279,70.937500
3,batch2,untreated,batch2_1485_unt,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,H3.3 > 5 and H4 > 5 on raw values,5533,4015,1518,72.564612
4,batch2,treated,batch2_1494_gem,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,H3.3 > 5 and H4 > 5 on raw values,2705,1923,782,71.090573
5,batch2,untreated,batch2_1610_unt,GFP_high,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,H3.3 > 5 and H4 > 5 on raw values,3987,2984,1003,74.843241
6,batch2,treated,batch2_1473_gem,GFP_low,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,H3.3 > 5 and H4 > 5 on raw values,10768,3468,7300,32.206538
7,batch2,untreated,batch2_1474_unt,GFP_low,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,H3.3 > 5 and H4 > 5 on raw values,15060,4997,10063,33.180611
8,batch2,treated,batch2_1484_gem,GFP_low,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,H3.3 > 5 and H4 > 5 on raw values,13361,4569,8792,34.196542
9,batch2,untreated,batch2_1485_unt,GFP_low,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,H3.3 | H4,H3.3 > 5 and H4 > 5 on raw values,16709,5107,11602,30.564367


## 4. Arcsinh transformation

In [8]:
ARCSINH_ROOT = OUTPUT_ROOT / "4. data processing after GFP split" / "ARCSINH_b"
arcsinh_summary = process_arcsinh_files(GATED_ROOT, ARCSINH_ROOT)
display(arcsinh_summary)

,input_gated_file,output_arcsinh_file,n_cells,n_columns_total,n_marker_columns_transformed,transformed_columns
0,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,1638,54,40,CD109 | CD24 | CD44 | CD45 | CXCR4 | DNA | E-c...
1,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,681,54,40,CD109 | CD24 | CD44 | CD45 | CXCR4 | DNA | E-c...
2,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,1923,54,40,CD109 | CD24 | CD44 | CD45 | CXCR4 | DNA | E-c...
3,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,3399,54,40,CD109 | CD24 | CD44 | CD45 | CXCR4 | DNA | E-c...
4,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,4015,54,40,CD109 | CD24 | CD44 | CD45 | CXCR4 | DNA | E-c...
5,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,2984,54,40,CD109 | CD24 | CD44 | CD45 | CXCR4 | DNA | E-c...
6,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,3468,54,40,CD109 | CD24 | CD44 | CD45 | CXCR4 | DNA | E-c...
7,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,4569,54,40,CD109 | CD24 | CD44 | CD45 | CXCR4 | DNA | E-c...
8,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,3155,54,40,CD109 | CD24 | CD44 | CD45 | CXCR4 | DNA | E-c...
9,/Users/yishai/Desktop/epigenetics_field_theory...,/Users/yishai/Desktop/epigenetics_field_theory...,4997,54,40,CD109 | CD24 | CD44 | CD45 | CXCR4 | DNA | E-c...


## 5. Inspect identities, metadata, and marker values

In [9]:
count_rows = []
for path in sorted(ARCSINH_ROOT.rglob("*_ARCSINH_5.csv")):
    frame = pd.read_csv(path, low_memory=False)
    count_rows.append(
        {
            "experiment": path.relative_to(ARCSINH_ROOT).parts[0],
            "gfp_group": path.relative_to(ARCSINH_ROOT).parts[1],
            "sample_name": frame["sample_name"].iloc[0] if len(frame) else "",
            "n_cells": len(frame),
            "unique_cell_ids": frame["cell_id"].nunique(),
            "metadata_columns_present": all(
                column in frame
                for column in [
                    "experiment",
                    "condition",
                    "replicate",
                    "source_file",
                    "source_path",
                    "raw_row_id",
                    "Line",
                    "batch",
                ]
            ),
            "marker_columns": sum(
                column
                not in [
                    "cell_id",
                    "experiment",
                    "sample_name",
                    "condition",
                    "replicate",
                    "source_file",
                    "source_path",
                    "raw_row_id",
                    "Line",
                    "batch",
                ]
                for column in frame.columns
            ),
        }
    )
inspection_summary = pd.DataFrame(count_rows)
display(inspection_summary)
if (
    not inspection_summary.empty
    and not (inspection_summary["n_cells"] == inspection_summary["unique_cell_ids"]).all()
):
    raise AssertionError("Cell IDs are not unique in one or more arcsinh outputs")
if any(
    column.startswith("Unnamed:")
    for path in ARCSINH_ROOT.rglob("*.csv")
    for column in pd.read_csv(path, nrows=0).columns
):
    raise AssertionError("Unexpected index artifact column in an arcsinh output")

,experiment,gfp_group,sample_name,n_cells,unique_cell_ids,metadata_columns_present,marker_columns
0,batch2,GFP_high,batch2_1473_gem,1638,1638,True,44
1,batch2,GFP_high,batch2_1484_gem,681,681,True,44
2,batch2,GFP_high,batch2_1494_gem,1923,1923,True,44
3,batch2,GFP_high,batch2_1474_unt,3399,3399,True,44
4,batch2,GFP_high,batch2_1485_unt,4015,4015,True,44
5,batch2,GFP_high,batch2_1610_unt,2984,2984,True,44
6,batch2,GFP_low,batch2_1473_gem,3468,3468,True,44
7,batch2,GFP_low,batch2_1484_gem,4569,4569,True,44
8,batch2,GFP_low,batch2_1494_gem,3155,3155,True,44
9,batch2,GFP_low,batch2_1474_unt,4997,4997,True,44
